<a href="https://colab.research.google.com/github/massalina/ohrana_truda/blob/main/proverka_reg_num_mintrud.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np

# Загружаем файлы
df1 = pd.read_csv('Протоколы_проверка.csv', sep=';', encoding='utf-8', low_memory=False)
df2 = pd.read_csv('Реестр_обученных.csv', sep=';', encoding='utf-8', low_memory=False)

print("📂 Файлы загружены")

# Удаляем строки с пустыми ключевыми данными
df1 = df1.dropna(subset=['СНИЛС', 'Фамилия', 'Имя'])
df2 = df2.dropna(subset=['СНИЛС', 'Фамилия', 'Имя'])

print(f"📊 Первый файл: {len(df1)} строк, Второй файл: {len(df2)} строк")

# Сохраняем оригинальные данные
df1_orig = df1.copy()
df2_orig = df2.copy()

# Поля для ключа (БЕЗ номера протокола)
key_fields = ['Фамилия', 'Имя', 'Отчество', 'СНИЛС', 'Должность',
              'Дата проверки знаний', 'Программа обучения']

# Очищаем данные для создания ключа
for col in key_fields:
    if col in df1.columns:
        df1[col + '_clean'] = df1[col].astype(str).str.strip().str.lower()
    if col in df2.columns:
        df2[col + '_clean'] = df2[col].astype(str).str.strip().str.lower()

# Создаём ключи
df1['key'] = df1[[col + '_clean' for col in key_fields]].astype(str).agg('|'.join, axis=1)
df2['key'] = df2[[col + '_clean' for col in key_fields]].astype(str).agg('|'.join, axis=1)

# Находим столбец с регистрационным номером
reg_column = None
for col in df2.columns:
    if 'номер в реестре' in col.lower() or 'регистр' in col.lower() or 'номер' in col.lower():
        reg_column = col
        break

if reg_column is None:
    reg_column = df2.columns[0]

print(f"✅ Столбец с регистрационным номером: '{reg_column}'")

# Объединяем таблицы
print("🔄 Объединение таблиц...")
df1_orig_with_key = df1_orig.copy()
df1_orig_with_key['key'] = df1['key'].values

result = df1_orig_with_key.merge(df2[['key', reg_column]], on='key', how='left')

# Находим несовпадающие строки
not_found = result[result[reg_column].isna()].copy()

total = len(df1_orig)
found = total - len(not_found)

print(f"\n📊 Статистика:")
print(f"  Всего строк в первом файле: {total}")
print(f"  ✅ Найдено совпадений: {found}")
print(f"  ❌ Не найдено соответствий: {len(not_found)}")

# Если есть несовпадающие строки
if len(not_found) > 0:
    print("\n🔄 Анализ несовпадающих строк...")

    # Создаём словарь для быстрого поиска по СНИЛС + Программа
    df2_dict = {}
    for _, row in df2_orig.iterrows():
        snils = str(row['СНИЛС']).strip().lower()
        prog = str(row['Программа обучения']).strip().lower()
        key = (snils, prog)
        if key not in df2_dict:
            df2_dict[key] = []
        df2_dict[key].append(row)

    print(f"✅ Создан словарь для быстрого поиска ({len(df2_dict)} уникальных комбинаций)")

    # Функция для поиска расхождений
    def find_differences_with_critical(row, df2_dict, key_fields, reg_column):
        """Находит расхождения и определяет критичность"""
        snils = str(row['СНИЛС']).strip().lower()
        prog = str(row['Программа обучения']).strip().lower()
        search_key = (snils, prog)

        result = []
        if search_key in df2_dict:
            for df2_row in df2_dict[search_key]:
                differences = {}
                matched_fields = []
                critical_issues = []

                for field in key_fields:
                    if field in row and field in df2_row:
                        val1 = str(row[field]).strip()
                        val2 = str(df2_row[field]).strip()

                        if val1.lower() == val2.lower():
                            matched_fields.append(field)
                        else:
                            differences[field] = {
                                'в_первом_файле': val1,
                                'во_втором_файле': val2
                            }
                            if field == 'Дата проверки знаний':
                                critical_issues.append(field)

                match_percent = len(matched_fields) / len(key_fields) * 100

                if critical_issues:
                    status = 'КРИТИЧЕСКОЕ РАСХОЖДЕНИЕ (дата не совпадает!)'
                elif differences:
                    status = 'Некритическое расхождение'
                else:
                    status = 'Полное совпадение (ошибка в ключе?)'

                result.append({
                    'номер_в_реестре': df2_row[reg_column],
                    'всего_полей': len(key_fields),
                    'совпадает_полей': len(matched_fields),
                    'процент_совпадения': round(match_percent, 1),
                    'совпадающие_поля': ', '.join(matched_fields) if matched_fields else 'нет',
                    'расхождения': differences,
                    'critical_issues': critical_issues,
                    'status': status
                })

        return result

    # Применяем анализ
    print("🔄 Поиск расхождений...")
    not_found['анализ'] = not_found.apply(
        lambda row: find_differences_with_critical(row, df2_dict, key_fields, reg_column), axis=1
    )

    # Определяем статус строки
    def get_status(analysis):
        if not analysis:
            return 'Нет похожих записей'

        critical_matches = [a for a in analysis if 'КРИТИЧЕСКОЕ' in a['status']]
        if critical_matches:
            best = critical_matches[0]
            diff_fields = list(best['расхождения'].keys())
            return f"⚠️ КРИТИЧЕСКОЕ! Расхождения: {', '.join(diff_fields)}"

        best_match = max(analysis, key=lambda x: x['процент_совпадения'])
        if best_match['процент_совпадения'] == 100:
            return 'Полное совпадение'
        else:
            diff_fields = list(best_match['расхождения'].keys())
            return f"Некритическое расхождение ({best_match['процент_совпадения']}%). Поля: {', '.join(diff_fields)}"

    not_found['статус'] = not_found['анализ'].apply(get_status)

    # 🔥 СОЗДАЁМ ИТОГОВЫЙ ФАЙЛ (БЕЗ дублирующего столбца)
    print("\n📝 Создание итогового файла...")

    export_columns = ['Фамилия', 'Имя', 'Отчество', 'СНИЛС', 'Должность',
                      'Номер протокола', 'Дата проверки знаний', 'Программа обучения']

    export_data = []
    for idx, row in not_found.iterrows():
        base_data = {col: row[col] for col in export_columns}
        base_data['статус'] = row['статус']  # 🔥 Только статус, без критическое_расхождение

        if row['анализ']:
            critical = [a for a in row['анализ'] if 'КРИТИЧЕСКОЕ' in a['status']]
            if critical:
                best_match = critical[0]
            else:
                best_match = max(row['анализ'], key=lambda x: x['процент_совпадения'])

            base_data['номер_в_реестре'] = best_match['номер_в_реестре']
            base_data['процент_совпадения'] = best_match['процент_совпадения']

            if best_match['расхождения']:
                diff_text = []
                for field, vals in best_match['расхождения'].items():
                    critical_marker = ' ⚠️ КРИТИЧЕСКИЙ!' if field == 'Дата проверки знаний' else ''
                    diff_text.append(f"{field}{critical_marker}: '{vals['в_первом_файле']}' ≠ '{vals['во_втором_файле']}'")
                base_data['расхождения'] = '; '.join(diff_text)
            else:
                base_data['расхождения'] = 'Нет расхождений'
        else:
            base_data['номер_в_реестре'] = 'Нет'
            base_data['процент_совпадения'] = 0
            base_data['расхождения'] = 'Нет похожих записей'

        export_data.append(base_data)

    export_df = pd.DataFrame(export_data)

    # 🔥 СОРТИРУЕМ по статусу: сначала критические
    export_df['is_critical'] = export_df['статус'].str.contains('КРИТИЧЕСКОЕ', case=False, na=False)
    export_df = export_df.sort_values('is_critical', ascending=False)
    export_df = export_df.drop(columns=['is_critical'])

    # Сохраняем
    export_df.to_csv('несовпадающие_строки_полный_отчет.csv', sep=';', index=False, encoding='utf-8-sig')

    # Статистика
    critical_count = export_df[export_df['статус'].str.contains('КРИТИЧЕСКОЕ', case=False, na=False)].shape[0]
    no_match_count = export_df[export_df['статус'] == 'Нет похожих записей'].shape[0]
    other_count = len(export_df) - critical_count - no_match_count

    print("\n" + "="*60)
    print("📊 ИТОГОВАЯ СТАТИСТИКА")
    print("="*60)
    print(f"  ❌ Всего не найдено соответствий: {len(not_found)}")
    print(f"  ⚠️ Из них с КРИТИЧЕСКИМ расхождением (дата не совпадает): {critical_count}")
    print(f"  ℹ️ Некритические расхождения: {other_count}")
    print(f"  ❓ Нет похожих записей во втором файле: {no_match_count}")
    print("="*60)
    print(f"\n✅ Создан файл 'несовпадающие_строки_полный_отчет.csv'")
    print(f"   Всего строк в файле: {len(export_df)}")
    print("\n📌 Файл отсортирован: сначала критические расхождения")
    print("📌 В Excel вы можете отфильтровать столбец 'статус' для поиска критических записей")

else:
    print("\n🎉 Все строки найдены! Нет несовпадающих записей.")


📂 Файлы загружены
📊 Первый файл: 999 строк, Второй файл: 42469 строк
✅ Столбец с регистрационным номером: 'Номер в реестре'
🔄 Объединение таблиц...

📊 Статистика:
  Всего строк в первом файле: 999
  ✅ Найдено совпадений: 541
  ❌ Не найдено соответствий: 458

🔄 Анализ несовпадающих строк...
✅ Создан словарь для быстрого поиска (29874 уникальных комбинаций)
🔄 Поиск расхождений...

📝 Создание итогового файла...

📊 ИТОГОВАЯ СТАТИСТИКА
  ❌ Всего не найдено соответствий: 458
  ⚠️ Из них с КРИТИЧЕСКИМ расхождением (дата не совпадает): 100
  ℹ️ Некритические расхождения: 0
  ❓ Нет похожих записей во втором файле: 358

✅ Создан файл 'несовпадающие_строки_полный_отчет.csv'
   Всего строк в файле: 458

📌 Файл отсортирован: сначала критические расхождения
📌 В Excel вы можете отфильтровать столбец 'статус' для поиска критических записей
